# Part 1: What is Machine Learning?
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Sarah's 10,000 reviews

**Sarah Chen** was promoted to Customer Experience Analyst at **NorthStar Retail**, a mid-size online clothing store, at the start of the month. Early in her second week, her manager **Priya** drops a folder on her desk:

> "By Friday, I need to know which of these 10,000 customer reviews are *positive* and which are *negative*. Hand-reading them would take a full work-week. Figure something out."

**By the end of this section you will:**
- Compare two very different approaches (rule-based vs ML) on the same task
- Run a pre-trained ML model end-to-end on real reviews
- Be able to explain, in plain English, what Machine Learning actually *is*

## Where does Machine Learning fit? — a bridge from M1 and M2

You've now finished two modules. It is worth a 30-second pause to be clear about what changes in this one.

| Module | What you do with data | Sarah's version of the task |
|---|---|---|
| **M1 · Data Analytics** | **Summarise the past.** Describe what happened. | *"Reviews dropped 12% last month."* |
| **M2 · Data Engineering** | **Move and store data reliably.** Build the pipes. | *"Reviews now land in the warehouse every night."* |
| **M3 · Machine Learning** | **Predict the unseen.** Label or forecast things no human has looked at yet. | *"Here are 10,000 reviews we've never read — label each positive or negative."* |

**The shift in M3:** instead of describing data that already has answers, you are asking a computer to produce answers for data the team has never seen.

Everything else — SQL, Python, statistics — you are already carrying with you. What's *new* in M3 is **learning from examples** so the computer can handle work at a scale no team could do by hand. That is exactly what Sarah needs for 10,000 reviews by Friday.

---

## Setup

Run this cell first. It imports the libraries we'll use and confirms everything is installed.

In [1]:
import os
# Must be set before importing transformers to prevent tokenizer deadlock on macOS
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
from transformers import pipeline
import warnings

# macOS guard: cap PyTorch threads to 1 to prevent the OpenMP scheduler
# from hanging the sentiment-analysis loop. Harmless on Linux/Windows.
import torch
torch.set_num_threads(1)

# Hugging Face prints internal deprecation notes that are not errors.
# We silence them here so the output stays clean for learning.
warnings.filterwarnings("ignore")

print("✅ Libraries loaded — you're ready to go!")

✅ Libraries loaded — you're ready to go!


## Meet the reviews

Here are 5 sample reviews from NorthStar Retail. We'll use these to compare the two approaches.

In [2]:
reviews = [
    "Absolutely love this dress! Fabric is soft and the fit is perfect.",
    "Quality is cheap and the stitching came apart after one wash. Disappointed.",
    "It's okay. Nothing wrong with it but nothing special either.",
    "Terrible shipping — arrived 3 weeks late. Product itself is fine though.",
    "Not what I expected from the photo but actually really nice in person!",
]

for i, r in enumerate(reviews, 1):
    print(f"{i}. {r}")

1. Absolutely love this dress! Fabric is soft and the fit is perfect.
2. Quality is cheap and the stitching came apart after one wash. Disappointed.
3. It's okay. Nothing wrong with it but nothing special either.
4. Terrible shipping — arrived 3 weeks late. Product itself is fine though.
5. Not what I expected from the photo but actually really nice in person!


## Approach 1 — The traditional programming way

Sarah's first instinct is to write a rule-based program: *"Look for positive words, look for negative words, tally them up."*

Let's try it.

In [3]:
positive_words = ["love", "perfect", "great", "nice", "soft", "amazing", "excellent"]
negative_words = ["terrible", "cheap", "disappointed", "bad", "awful", "late", "broken"]


def rule_based_sentiment(review):
    text = review.lower()
    pos_hits = sum(1 for w in positive_words if w in text)
    neg_hits = sum(1 for w in negative_words if w in text)
    if pos_hits > neg_hits:
        return "POSITIVE"
    elif neg_hits > pos_hits:
        return "NEGATIVE"
    else:
        return "NEUTRAL"


# Let's see its verdict on our 5 reviews
for r in reviews:
    print(f"{rule_based_sentiment(r):>9} | {r}")

 POSITIVE | Absolutely love this dress! Fabric is soft and the fit is perfect.
 NEGATIVE | Quality is cheap and the stitching came apart after one wash. Disappointed.
  NEUTRAL | It's okay. Nothing wrong with it but nothing special either.
 NEGATIVE | Terrible shipping — arrived 3 weeks late. Product itself is fine though.
 POSITIVE | Not what I expected from the photo but actually really nice in person!


## Approach 2 — The Machine Learning way

Instead of writing rules by hand, we let a model learn from millions of labelled reviews what "positive" and "negative" *actually look like* in real language.

The good news: **we don't have to train this model ourselves.** Someone at Hugging Face already trained it on millions of examples. We just load it and use it.

This is called using a **pre-trained model**.

In [4]:
# Load a pre-trained sentiment model
# First run: downloads ~250MB the first time, then cached
print("Loading pre-trained sentiment model... (first run may take ~1 min)")
ml_sentiment = pipeline("sentiment-analysis")
print("✅ Model ready.")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading pre-trained sentiment model... (first run may take ~1 min)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

✅ Model ready.


### ⏸️ Sarah's check-in

The pre-trained ML model is loaded. Before Sarah runs it on the same 5 reviews she just tested with rules, she pauses:

- *"Will this actually do better than my word-counting rules?"*
- *"Where will it still struggle — the mixed ones? The short ones?"*

Form your own prediction before running the next cell. Even a rough guess makes the output much more meaningful.

In [5]:
for r in reviews:
    result = ml_sentiment(r)[0]
    label = result["label"]
    confidence = result["score"]
    print(f"{label:>9} ({confidence:.2%}) | {r}")

 POSITIVE (99.99%) | Absolutely love this dress! Fabric is soft and the fit is perfect.
 NEGATIVE (99.94%) | Quality is cheap and the stitching came apart after one wash. Disappointed.
 NEGATIVE (93.00%) | It's okay. Nothing wrong with it but nothing special either.
 POSITIVE (98.34%) | Terrible shipping — arrived 3 weeks late. Product itself is fine though.
 POSITIVE (99.98%) | Not what I expected from the photo but actually really nice in person!


### 💡 What do you notice — Rules vs ML?

Compare the rule-based output (5 reviews ago) with the ML output (just now). A few patterns:

**Where rules struggled:**
- **Review #4** ("Terrible shipping — product itself is fine") got labelled NEGATIVE by the rules, but the product *is* positive. Rules can't tell what a word refers to.
- Extending the rule list quickly turns into whack-a-mole — every new review surfaces a new edge case.

**Where ML did better:**
- **Review #4** is handled in context, not as a bag of isolated words.
- Each prediction comes with a **confidence score** — the model says "POSITIVE with 97% confidence" not just "POSITIVE." Lower confidence = a signal to route a case to a human reviewer.
- On ambiguous reviews like #3 ("it's okay"), the model picks one label but with a lower score — exactly the kind of case that deserves human review.

**Back to our scenario:**
> Sarah can now classify all 10,000 reviews in an hour using the pre-trained model. She can also filter to "NEGATIVE with >90% confidence" to focus Priya's attention on the most confidently-bad cases — something rules couldn't easily give her.

## So what just happened?

### Traditional programming vs Machine Learning

| Dimension | Traditional programming | Machine Learning |
|---|---|---|
| **Input** | Rules + data | Examples (data + correct answers) |
| **Programmer writes** | The rules | The data collection + training setup |
| **Output** | Deterministic answer | Probabilistic prediction + confidence |
| **Handles new cases?** | Only if the rule covered it | Yes, if the pattern is similar to training data |
| **Failure mode** | Obvious (rule didn't fire) | Subtle (model is confidently wrong on edge cases) |

### The ML mental model

> *"In traditional programming, you write the **rules**. In Machine Learning, you write the **examples**, and the computer discovers the **rules**."*

This one idea explains every technique in the rest of the course — regression, trees, neural networks, GenAI. They are all different ways of **learning rules from examples**.

## 📎 Aside — Why a deep-learning model, and not a lexicon like TextBlob?

You may encounter **TextBlob** — another popular Python sentiment library — elsewhere in tutorials. It *looks* like ML, but under the hood it is essentially a **lexicon**: a long dictionary mapping individual words to fixed polarity scores. For straightforward text it works fine; for natural language with negation, sarcasm, or domain-specific words, it fails in confident, surprising ways.

Here is one of the complaints Sarah meets in the assignment notebook:

> *"Absolutely furious. This app has deleted my transaction history twice."*

| Model | Verdict | Why |
|---|---|---|
| **TextBlob** (lexicon) | **POSITIVE** (polarity `+0.20`) | "furious" is **missing from its word list**; "absolutely" is scored positive in isolation, so the modifier wins. |
| **DistilBERT** (deep learning) | **NEGATIVE** (≥ 99% confidence) | The model learned the meaning of "furious" *in context* from millions of examples — including the negative cue from "deleted my transaction history." |

This is why the rest of this lesson — and the assignment — uses the DistilBERT pipeline rather than a lexicon-based tool, even for what looks like a simple task.

> **The learning point:** Choosing the *right kind* of model for a task is often more important than tuning a model you've already chosen. A great deep-learning model beats a poor lexicon. A great lexicon (with hand-curated rules for your domain) beats a poorly-trained deep-learning model. Don't reach for a tool because it's familiar — reach for the one whose failure modes you can live with.

---

## ✅ Section 1 Summary

| Concept | What it means | Real-world use |
|---|---|---|
| **Traditional programming** | You write the rules; computer follows them | Tax calculations, unit conversions, data validation |
| **Machine Learning** | You provide examples; computer learns the rules | Sentiment analysis, fraud detection, recommendations |
| **Pre-trained model** | A model someone else already trained you can use directly | Sarah's sentiment classifier — no labelled data needed from her |
| **Confidence score** | How sure the model is about each prediction | Routing low-confidence cases to a human reviewer |

**Key insight for our scenario:**
> Sarah can classify 10,000 reviews in an hour by loading someone else's pre-trained model — but she still needs to decide when to trust its output and when to escalate to a human.

---
**Up next → Section 2:** The three *kinds* of Machine Learning, and how to tell which one fits a given business problem.
Open `03_three_categories.ipynb`

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

This optional section covers:
- **Sarah's rule-based check-in** — an extra Pause-and-Predict on the rule-based output
- **Scaling up** — running the ML model on 20 more reviews and inspecting confidence
- **A quick stats lens** — descriptive statistics on review length and sentiment (foreshadowing L02)
- **Reflection questions** — three open questions to take into the assignment

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · January 2023.*
> This is Chapter 1 of the NorthStar Retail story — the morning Sarah received the USB drive. If you haven't run `01_monday_morning.ipynb` yet, do that first. This notebook goes deeper.

### ⏸️ Sarah's check-in

Sarah scrolls through the rule-based output and frowns: *"Some of these look right. Some look plainly dumb."*

Before looking at the answers below, do what Sarah would do — look at your own output and ask yourself:

- Which reviews did the rules get wrong?
- *Why* did they get them wrong?
- What would you need to add to make the rules better?

*Double-click this cell and jot down one concrete weakness you can see. Press Shift+Enter to render it back.*

## Running it on more reviews (the whole point)

Let's simulate Sarah's real workflow. Here are 20 more reviews.

In [6]:
more_reviews = [
    "Arrived quickly, exactly as described. Happy!",
    "The sweater's colour is way off from the photo.",
    "Five stars — best jeans I own.",
    "Seams unraveling after two washes. Won't buy again.",
    "Fit is tight but that's on me, not the product.",
    "Absolutely gorgeous, so many compliments!",
    "Returned it. Not worth the price.",
    "Decent quality, good price point.",
    "My daughter adores this jacket. 10/10.",
    "The fabric pills after one wash. Very disappointing.",
    "Perfect fit, great quality. Will buy again.",
    "Ugly in person. Don't trust the photos.",
    "I've worn this every week since I got it. Love it.",
    "Too small — sizing runs 2 sizes under.",
    "Great gift for my wife, she loved it.",
    "Lost a button in a week. Poor quality control.",
    "Arrived with a tag from a different brand attached — weird but product is fine.",
    "Exactly what I was looking for, finally a store that gets it right.",
    "Shipping took forever. Product is okay, not amazing.",
    "Returned, refunded quickly. Customer service was kind.",
]

results = ml_sentiment(more_reviews)
df = pd.DataFrame({
    "review": more_reviews,
    "sentiment": [r["label"] for r in results],
    "confidence": [r["score"] for r in results],
})
df = df[["sentiment", "confidence", "review"]]
df.head(10)

,sentiment,confidence,review
0,POSITIVE,0.999860,"Arrived quickly, exactly as described. Happy!"
1,NEGATIVE,0.999376,The sweater's colour is way off from the photo.
2,POSITIVE,0.998436,Five stars — best jeans I own.
3,NEGATIVE,0.996658,Seams unraveling after two washes. Won't buy a...
4,NEGATIVE,0.995991,"Fit is tight but that's on me, not the product."
5,POSITIVE,0.999884,"Absolutely gorgeous, so many compliments!"
6,NEGATIVE,0.999699,Returned it. Not worth the price.
7,POSITIVE,0.999854,"Decent quality, good price point."
8,POSITIVE,0.997113,My daughter adores this jacket. 10/10.
9,NEGATIVE,0.999780,The fabric pills after one wash. Very disappoi...


### Sanity-checking the output

In [7]:
# How confident is the model on average?
print(f"Average confidence:      {df['confidence'].mean():.2%}")
print(f"Lowest confidence:       {df['confidence'].min():.2%}")

# How does it split positive vs negative?
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

# Which reviews is it least sure about?
# Use .2% so we can see the variance — these all round to 100% otherwise.
print("\nTop 3 least confident predictions (worth human review):")
for _, row in df.nsmallest(3, "confidence").iterrows():
    print(f"  {row['sentiment']} ({row['confidence']:.2%}): {row['review']}")

Average confidence:      99.91%
Lowest confidence:       99.60%

Sentiment distribution:
sentiment
POSITIVE    11
NEGATIVE     9
Name: count, dtype: int64

Top 3 least confident predictions (worth human review):
  NEGATIVE (99.60%): Fit is tight but that's on me, not the product.
  NEGATIVE (99.67%): Seams unraveling after two washes. Won't buy again.
  POSITIVE (99.71%): My daughter adores this jacket. 10/10.


### 💡 What do you notice?

- The **average confidence is high**, which is typical for pre-trained sentiment models on straightforward text.
- The **least-confident predictions are exactly the ambiguous reviews** ("Shipping took forever. Product is okay, not amazing.") — the ones a human *should* look at.
- The model output is now a **dataframe** Sarah can filter, sort, and export. She can hand her manager a CSV by lunchtime.

**The big shift:** We went from *"this will take a week"* to *"this will take an hour, and I can flag ambiguous cases for human review"* — by using a model someone else already trained.

## A quick stats lens on the reviews

Before Sarah writes up the numbers, she takes a few minutes to *look* at the shape of her data. No formulas yet — just a feel for what's in there.

Three questions she wants a quick answer to:

1. **How long are typical reviews?** (Are most of them short "ok" / "good" notes, or long rants?)
2. **Is there variability — or does everyone write the same kind of review?**
3. **Is there any relationship between how *long* a review is and how *positive* it feels?**

We'll come back to these ideas formally in **L02 — Statistics**. For now, think of this as the kind of look you'd give a new dataset on any job.

In [8]:
# Three light-touch questions about the reviews.

# Add a review length column (characters) and a rough "positivity" score
# where POSITIVE = 1 and NEGATIVE = 0, weighted by the model's confidence.
df["length_chars"] = df["review"].str.len()
df["positivity"] = df.apply(
    lambda r: r["confidence"] if r["sentiment"] == "POSITIVE" else 1 - r["confidence"],
    axis=1,
)

# 1. Typical length
print("--- How long are the reviews? ---")
print(f"Shortest: {df['length_chars'].min()} characters")
print(f"Longest : {df['length_chars'].max()} characters")
print(f"Average : {df['length_chars'].mean():.0f} characters")
print(f"Spread (standard deviation): {df['length_chars'].std():.0f} characters")

# 2. Distribution — a quick histogram-in-text style summary
print()
print("--- Distribution of review lengths ---")
buckets = pd.cut(df["length_chars"], bins=[0, 40, 80, 120, 200], labels=["very short", "short", "medium", "long"])
print(buckets.value_counts().sort_index())

# 3. Any relationship between length and positivity?
correlation = df[["length_chars", "positivity"]].corr().iloc[0, 1]
print()
print(f"--- Relationship between length and positivity ---")
print(f"Correlation: {correlation:+.2f}")
print("(Closer to +1 means longer = more positive; closer to -1 means longer = more negative; near 0 means no clear link.)")

--- How long are the reviews? ---
Shortest: 30 characters
Longest : 79 characters
Average : 46 characters
Spread (standard deviation): 12 characters

--- Distribution of review lengths ---
length_chars
very short     7
short         13
medium         0
long           0
Name: count, dtype: int64

--- Relationship between length and positivity ---
Correlation: +0.09
(Closer to +1 means longer = more positive; closer to -1 means longer = more negative; near 0 means no clear link.)


### 💡 What Sarah notices

- **Most reviews are short** — usually well under 100 characters. Outliers stretch longer. That's a typical "skewed" shape we'll put a name on in L02.
- **The spread matters.** Some reviews are one word, others are two paragraphs. That variability is why an average alone is a weak summary of a dataset.
- **The correlation is small either way** — on this tiny sample of 20 reviews, length doesn't strongly predict positivity. On 10,000 reviews, that number could still shift, and *that* would be worth a conversation with Aisha.

**What we are doing here is descriptive statistics** — looking at shape, centre, spread, and relationships to understand the data *before* modelling. We'll formalise every word in that sentence in L02. For now, just notice: even a quick 30-second scan can surface things that change how you build the model.

## Reflection questions

Take 3 minutes.

> **How to answer inside the notebook:** *double-click* the "*Your answers:*" cell below to edit it, type your answer, then press **Shift+Enter** to render it back to formatted text.

1. Name one problem from your own work or life where ML would clearly be the *wrong* tool — and say why.
2. The pre-trained model Sarah used was trained on general product reviews. Would you trust it for classifying *medical symptom* reports? Why or why not?
3. Sarah's manager wants to automatically send apology coupons to every review the model classifies as NEGATIVE. What is one concern you would raise before pressing the "send" button?

*Your answers:*

1.
2.
3.